# A Multidimensional Predictive and Clustering Approach to Understanding State-Level Chronic Disease Disparities

## Team 8 - CMPE 255 Spring 2026

**Team Members:**
- Bharath Kumar A [018221268]
- Mithil Sri Sai Devisetty [019153394]
- Mani Mokshith Noonety [019100458]
- Tej Kiran Yenugunti [019104878]

---

## Project Overview

This notebook implements a comprehensive data-driven pipeline to analyze chronic disease disparities across U.S. states. We integrate eight public health datasets and apply machine learning techniques to identify predictive factors and state archetypes.

## 1. Setup and Library Installation

In [ ]:
# Install gdown for Google Drive data downloads
!pip install gdown --quiet

## 2. Import Required Libraries

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Data downloading
import gdown

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Advanced ML
from xgboost import XGBRegressor

# Statistical analysis
from scipy import stats

# Settings
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All libraries imported successfully!")

## 3. Data Loading

We will load eight state-level datasets from public repositories.

### 3.1 Dataset 1: Chronic Disease (CDC BRFSS)

Contains state-level chronic disease metrics by year, topic, and gender.

In [ ]:
# Download Chronic Disease Dataset
print("Downloading Chronic Disease dataset...")
gdown_id = "1GA_camLQGvBYLLnPa9vcNQvfZHWBWJhN"
gdown.download(id=gdown_id, output="../data/ChronicDisease.csv", quiet=False)

# Load the dataset
df_chronic = pd.read_csv("../data/ChronicDisease.csv")

print(f"\n✓ Loaded Chronic Disease dataset: {df_chronic.shape}")
print(f"Columns: {df_chronic.shape[1]}")
print(f"Rows: {df_chronic.shape[0]}")

In [ ]:
# Initial exploration
print("Dataset Info:")
df_chronic.info()

print("\nFirst 5 rows:")
df_chronic.head()

In [ ]:
# Check for missing values
print("Missing values per column:")
df_chronic.isnull().sum()

### 3.2 Clean Chronic Disease Dataset

We'll clean the dataset by:
1. Removing unnecessary columns
2. Filtering relevant disease topics
3. Removing non-state entries (US aggregates, territories)
4. Handling missing values

In [ ]:
# Step 1: Drop unnecessary columns
cols_to_drop = [
    "Response", "DataValueAlt", "DataValueFootnoteSymbol", "DataValueFootnote",
    "LowConfidenceLimit", "HighConfidenceLimit",
    "StratificationCategory2", "Stratification2",
    "StratificationCategory3", "Stratification3",
    "StratificationCategoryID2", "StratificationID2",
    "StratificationCategoryID3", "StratificationID3",
    "ResponseID", "Geolocation", "LocationID", "TopicID", "QuestionID", "DataValueTypeID"
]

df_chronic_clean = df_chronic.drop(columns=[c for c in cols_to_drop if c in df_chronic.columns])

print(f"✓ Dropped {len([c for c in cols_to_drop if c in df_chronic.columns])} unnecessary columns")
print(f"Remaining columns: {df_chronic_clean.shape[1]}")

In [ ]:
# Step 2: Filter relevant disease topics (remove non-chronic disease topics)
topics_to_drop = [
    "Nutrition, Physical Activity, and Weight Status",
    "Health Status",
    "Social Determinants of Health",
    "Immunization",
    "Cognitive Health and Caregiving",
    "Maternal Health"
]

df_chronic_clean = df_chronic_clean[~df_chronic_clean["Topic"].isin(topics_to_drop)]

print(f"✓ Filtered to chronic disease topics only")
print(f"Remaining rows: {df_chronic_clean.shape[0]}")

In [ ]:
# Step 3: Remove rows with missing DataValue
df_chronic_clean = df_chronic_clean[df_chronic_clean["DataValue"].notna()]

print(f"✓ Removed rows with missing disease values")
print(f"Remaining rows: {df_chronic_clean.shape[0]}")

In [ ]:
# Step 4: Standardize data types and rename columns
df_chronic_clean["YearStart"] = df_chronic_clean["YearStart"].astype(int)
df_chronic_clean["YearEnd"] = df_chronic_clean["YearEnd"].astype(int)
df_chronic_clean["DataValue"] = pd.to_numeric(df_chronic_clean["DataValue"], errors="coerce")
df_chronic_clean["Year"] = df_chronic_clean["YearStart"]
df_chronic_clean = df_chronic_clean.rename(columns={"LocationDesc": "State"})

print("✓ Standardized data types and renamed columns")
print(f"\\nData types:")\nprint(df_chronic_clean.dtypes)

In [ ]:
# Step 5: Remove non-state entries (US aggregates, territories)
df_chronic_clean = df_chronic_clean[
    (df_chronic_clean["State"] != "United States") & 
    (df_chronic_clean["State"] != "Puerto Rico")
].reset_index(drop=True)

print(f"✓ Removed non-state entries")
print(f"Final cleaned dataset shape: {df_chronic_clean.shape}")
print(f"\\nUnique states: {df_chronic_clean['State'].nunique()}")
print(f"Unique disease topics: {df_chronic_clean['Topic'].nunique()}")

In [ ]:
# Display cleaned data sample
print("Cleaned Chronic Disease Dataset Sample:")
df_chronic_clean.head(10)

---

**Next Steps:**
- Load remaining 7 datasets
- Clean and preprocess each dataset
- Create master dataframe
- Perform EDA
- Build ML models